## Pipelines

### Relevant Information

Links: [Tutorial](https://www.kaggle.com/code/alexisbcook/pipelines), [Exercise](https://www.kaggle.com/code/marcopolo1313/exercise-pipelines/edit?fromFork=1), [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), [Pipelines](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)

+ The aim is to achieve bundelling of preprocessing and modelling steps. 
This leads to a cleaner code which is easier to productionize with greater options for model variation and lesser bugs.

+ ColumnTransform is basically concatenation of features.

+ Pipelines are vertical/ sequential

### Setup

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv(r'C:\Users\Anwesh\Desktop\Py\MachineLearning\KaggleML-Intermediate\DatasetsKaggleML-Intermediate\melb_data.csv')
y = data.Price
X = data.drop(['Price'], axis=1)

X_train_full, X_valid_full, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2,random_state=122)
list_of_missing_columns = [cols for cols in X_train_full.columns if X_train_full[cols].isnull().any()]
X_train_full.drop(list_of_missing_columns,axis=1) # inplace = True by default
X_valid_full.drop(list_of_missing_columns, axis=1)

#Remove High Cardinality Data
categorical_columns = [cols for cols in X_train_full.columns if X_train_full[cols].nunique()<10 and X_train_full[cols].dtype=='string']
#Select non-object, numerical Data
numerical_cols = [cols for cols in X_train_full.columns if X_train_full[cols].dtype in ['float64', 'int64']]

eligible_columns = categorical_columns + numerical_cols
X_train = X_train_full[eligible_columns].copy()
X_valid = X_valid_full[eligible_columns].copy()


In [19]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

#First we create separate preprocessing for numerical and categorical data
categorialData_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent'))
                                 ,('OHE_Object', OneHotEncoder(handle_unknown='ignore'))])
NumericalData_transformer = SimpleImputer(strategy='mean')

Bundeled_preprocessor = ColumnTransformer(transformers=[('categorical', categorialData_transformer,categorical_columns)
                                                        ,('numerical', NumericalData_transformer, numerical_cols)])


In [20]:
from sklearn.ensemble import RandomForestRegressor
ForestModel = RandomForestRegressor(n_estimators=180, random_state=232)

In [22]:
from sklearn.metrics import mean_absolute_error
myPipeline = Pipeline(steps=[('preprocessor', Bundeled_preprocessor)
                             ,('mymodel', ForestModel)])
myPipeline.fit(X_train, y_train)
preds = myPipeline.predict(X_valid)
score = mean_absolute_error(y_valid, preds)
print("Mean Absolute Error:",score) #using strategy mean for numerical data was better than mode or most_frequent. 
#mode gave MAE of around 156565

Mean Absolute Error: 156089.9751223399


## Exercise

### Setup

#### Basic Setup

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split

X_full = pd.read_csv(r'C:\Users\Anwesh\Desktop\Py\MachineLearning\KaggleML-Intermediate\DatasetsKaggleML-Intermediate\Pipelines\train.csv')
X_test_full = pd.read_csv(r'C:\Users\Anwesh\Desktop\Py\MachineLearning\KaggleML-Intermediate\DatasetsKaggleML-Intermediate\Pipelines\test.csv')

#separate predictors and features
X_full.dropna(axis=0, subset=['SalePrice']) #drop missing SalePrices first
y = X_full.SalePrice
X_full.drop(['SalePrice'],axis=1)

#train_test_split
X_train_full, X_valid_full, y_train, y_valid = train_test_split(X_full, y, train_size=0.8, test_size=0.2)

categorical_columns = [cols for cols in X_train_full.columns if X_train_full[cols].dtype == 'object' and X_train_full[cols].nunique()<10]
numerical_columns = [cols for cols in X_train_full.columns if X_train_full[cols].dtype in ['int64', 'float64']]
eligible_columns = categorical_columns + numerical_columns
X_train = X_train_full[eligible_columns].copy()
X_valid = X_valid_full[eligible_columns].copy()

#### Create Pipelines

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

#Preprocessing for numerical and categorical data, and pipeline for categorical data
numerical_transformer = SimpleImputer(strategy='constant', fill_value=1)
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent'))
                                          ,('OHE', OneHotEncoder(handle_unknown='ignore'))])

#Bundle
columnJoin = ColumnTransformer(transformers=[('numTransformer', numerical_transformer, numerical_columns)
                                             ,('catTransformer', categorical_transformer, categorical_columns)])

#Model
myModel = RandomForestRegressor(n_estimators=130, random_state=232)

#Create final PIPELINE
myPipeline = Pipeline(steps=[('joiner', columnJoin)
                             ,('forestModel', myModel)])
myPipeline.fit(X_train, y_train)
preds = myPipeline.predict(X_valid)
score = mean_absolute_error(y_valid, preds)